In [7]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

train_path = '/content/drive/MyDrive/Bitirme Projesi/train.csv'
test_path = '/content/drive/MyDrive/Bitirme Projesi/test.csv'

try:
    print("Loading and concatenating data (This process may take 15-20 seconds)...\n")
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # Map the 3 main domains of interest to their English names
    domain_map = {
        'urun_yorumlari': 'Product Reviews',
        'HUMIR': 'Movie Reviews',
        'tweet-pn': 'Tweets'
    }

    # Filter only these 3 domains and write their new names to the 'Domain' column
    df_filtered = df_all[df_all['dataset'].isin(domain_map.keys())].copy()
    df_filtered['Domain'] = df_filtered['dataset'].map(domain_map)

    missing_data = df_filtered[df_filtered['text'].isnull()].groupby('Domain').size().reset_index(name='Missing Values')

    duplicate_data = df_filtered[df_filtered.duplicated(subset=['text'], keep=False)].groupby('Domain').size().reset_index(name='Duplicate Reviews')

    # Create and merge the table
    domains_list = ['Product Reviews', 'Movie Reviews', 'Tweets']
    table2 = pd.DataFrame({'Domain': domains_list})
    table2 = pd.merge(table2, missing_data, on='Domain', how='left').fillna(0)
    table2 = pd.merge(table2, duplicate_data, on='Domain', how='left').fillna(0)

    # Convert counts to integers
    table2['Missing Values'] = table2['Missing Values'].astype(int)
    table2['Duplicate Reviews'] = table2['Duplicate Reviews'].astype(int)

    print("="*60)
    print("Data Quality Analysis")
    print("="*60)
    print(table2.to_string(index=False))
    print("\n")

    domain_texts = df_filtered.dropna(subset=['text']).groupby('Domain')['text'].apply(set).to_dict()

    # Calculate the intersections of the sets
    prod_movie = len(domain_texts.get('Product Reviews', set()).intersection(domain_texts.get('Movie Reviews', set())))
    prod_tweet = len(domain_texts.get('Product Reviews', set()).intersection(domain_texts.get('Tweets', set())))
    movie_tweet = len(domain_texts.get('Movie Reviews', set()).intersection(domain_texts.get('Tweets', set())))

    table3 = pd.DataFrame({
        'Domain Pair': ['Product-Movie', 'Product-Tweet', 'Movie-Tweet'],
        'Common Samples': [prod_movie, prod_tweet, movie_tweet]
    })

    print("="*60)
    print("Overlapping Samples")
    print("="*60)
    print(table3.to_string(index=False))
    print("\n")

    df_clean = df_filtered.dropna(subset=['text', 'label', 'dataset'])
    df_clean = df_clean.drop_duplicates(subset=['text'])
    df_binary = df_clean[df_clean['label'] != 'Notr']

    # Create the table (Group by Domain and Label)
    table1 = df_binary.groupby(['Domain', 'label']).size().unstack(fill_value=0)

    if 'Positive' in table1.columns and 'Negative' in table1.columns:
        table1 = table1[['Positive', 'Negative']]

    table1['Total'] = table1.sum(axis=1)

    table1 = table1.reset_index()

    print("="*60)
    print("Domain and Label Distribution (Cleaned Data)")
    print("="*60)
    print(table1.to_string(index=False))

except FileNotFoundError:
    print("ERROR: Files not found!")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading and concatenating data (This process may take 15-20 seconds)...

Data Quality Analysis
         Domain  Missing Values  Duplicate Reviews
Product Reviews               0                151
  Movie Reviews               0                920
         Tweets               0                  6


Overlapping Samples
  Domain Pair  Common Samples
Product-Movie               2
Product-Tweet               2
  Movie-Tweet               0


Domain and Label Distribution (Cleaned Data)
         Domain  Positive  Negative  Total
  Movie Reviews     32165     32295  64460
Product Reviews    219296     14810 234106
         Tweets      6087      4966  11053
